In [3]:
import pandas as pd

df22 = pd.read_csv('가구DB.csv')

In [7]:
df22.isnull().sum()

Unnamed: 0    0
goods_id      0
big_cat       0
small_cat     0
goods_name    1
price         1
Image URL     1
dtype: int64

In [ ]:
import pandas as pd
import boto3
import requests
from urllib.parse import urlparse
import os
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

# AWS 인증 정보
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
bucket_name = os.getenv("AWS_S3_BUCKET")
region_name = os.getenv("AWS_REGION")

# S3 클라이언트 생성
s3 = boto3.client(
    's3',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name
)

# ✅ 1. CSV 파일에서 이미지 URL 10개 가져오기
df = pd.read_csv('가구DB.csv')

# 이미지 URL과 Seq 있는 행 10개만 가져오기
df_selected = df[['goods_id', 'Image URL']].dropna()

# ✅ 3. goods_id 별 이미지 업로드
for goods_id, group in df_selected.groupby('goods_id'):
    group = group.reset_index(drop=True)

    for _, row in group.iterrows():
        url = row['Image URL']
        
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()

            parsed_url = urlparse(url)
            ext = os.path.splitext(parsed_url.path)[1] or '.jpg'

            file_name = f"{goods_id}{ext}"
            s3_key = f"furniture_image/{file_name}"

            s3.put_object(
                Bucket=bucket_name,
                Key=s3_key,
                Body=response.content,
                ContentType=response.headers.get('Content-Type', 'image/jpeg')
            )

            print(f"Uploaded: {s3_key}")

        except Exception as e:
            print(f"Failed to upload {url}: {e}")